# 00 · Go/No-Go for X-Pool on Colab
Runtime: **A100, High-RAM**. Purpose: measure, before the proposal (11/15), whether a faithful full-scale X-Pool reproduction fits our compute.

**Decision rule** (also written to `go_no_go.json`):
- official 9k checkpoint re-evaluated with our pipeline: |R@1 − 46.9| ≤ 1.0, and
- estimated full training run (5 epochs, 50 evals) ≤ 6 h on this GPU

→ **GO** with full MSR-VTT-9k. Otherwise see docs/ROADMAP.md §Fallbacks (7k split / plan B).

Expected wall time: ~1–1.5 h the first time (download dominates).

In [ ]:
# ==== 0. Settings (edit if needed) ====
DRIVE_ROOT = "/content/drive/MyDrive/cv_hw2"      # persistent: archive copy, checkpoints, runs
REPO_URL   = "https://github.com/beblack127/computervision.git"
BRANCH     = "main"
REPO       = "/content/computervision"
LOCAL_DATA = "/content/data/msrvtt"                # fast local disk (wiped when runtime ends)
CACHE_DIR  = "/content/cache/msrvtt_u12"           # frame cache (uniform 12 frames, raw uint8)
NUM_WORKERS = 8                                    # official default; keep fixed across runs

from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, time
os.makedirs(DRIVE_ROOT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1


In [ ]:
# ==== 1. Code: repo + pinned official X-Pool + deps ====
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -r requirements-colab.txt
!bash scripts/setup_xpool.sh
import sys; sys.path.insert(0, f"{REPO}/src")
!PYTHONPATH=src python -m pytest -q tests/ 2>&1 | tail -3


In [ ]:
# ==== 2. MSR-VTT videos (first time: download ~6.5GB and copy to Drive; later: copy from Drive) ====
t = time.time()
!python scripts/prepare_msrvtt.py --drive_dir {DRIVE_ROOT}/data --local_dir {LOCAL_DATA}
print(f"data ready in {time.time()-t:.0f}s")


In [ ]:
# ==== 3. Frame cache (bit-identical to official decoding) + verification on real videos ====
t = time.time()
!PYTHONPATH=src python -m tvr.frame_cache build --videos_dir {LOCAL_DATA}/videos --cache_dir {CACHE_DIR}
!PYTHONPATH=src python -m tvr.frame_cache stats --cache_dir {CACHE_DIR}
!PYTHONPATH=src python -m tvr.frame_cache verify --videos_dir {LOCAL_DATA}/videos --cache_dir {CACHE_DIR} --n 50
print(f"cache step: {time.time()-t:.0f}s")
!du -sh {CACHE_DIR}
!df -h /content | tail -1


In [ ]:
# ==== 4. Official checkpoint -> our evaluation pipeline (expect t2v R@1 ≈ 46.9) ====
!bash scripts/download_checkpoints.sh {DRIVE_ROOT}/checkpoints
!PYTHONPATH=src python -m tvr.evaluate --exp_name official_9k_eval --huggingface --dataset_name MSRVTT --msrvtt_train_file 9k --cache_dir {CACHE_DIR} --num_workers {NUM_WORKERS} --batch_size 32 --output_dir {DRIVE_ROOT}/runs --ckpt {DRIVE_ROOT}/checkpoints/official_9k/model_best.pth --dump_dir {DRIVE_ROOT}/runs/official_9k_eval/dump


In [ ]:
# ==== 5. Speed benchmark: 200 real training steps (no evals) + 1 eval ====
!PYTHONPATH=src python -m tvr.train --exp_name bench_9k --huggingface --dataset_name MSRVTT --msrvtt_train_file 9k --cache_dir {CACHE_DIR} --num_workers {NUM_WORKERS} --batch_size 32 --noclip_lr 3e-5 --transformer_dropout 0.3 --output_dir {DRIVE_ROOT}/runs --max_steps 200 --evals_per_epoch 0 --no_tensorboard


In [ ]:
# ==== 6. Summary + decision ====
ev = json.load(open(f'{DRIVE_ROOT}/runs/official_9k_eval/dump/eval_original.json'))
bm = json.load(open(f'{DRIVE_ROOT}/runs/bench_9k/run_meta.json'))
prep = json.load(open(f'{LOCAL_DATA}/prepare_report.json'))
cache = json.load(open(f'{CACHE_DIR}/index.json'))
r1 = ev['t2v']['R1']; hours = bm['benchmark']['estimated_full_run_hours']
summary = {
  'gpu': bm['env'].get('gpu_name'), 'gpu_mem_GB': bm['env'].get('gpu_total_memory_GB'),
  'cpu_count': bm['env'].get('cpu_count'), 'ram_GB': bm['env'].get('ram_total_GB'),
  'data_source': prep['source'], 'n_videos': prep['n_unique_ids'], 'n_missing': prep['n_missing'],
  'cache_build_seconds': cache['build_seconds'], 'cache_GB': round(cache['total_bytes_raw']/1e9, 1),
  'official_ckpt_t2v': ev['t2v'], 'official_ckpt_v2t': ev['v2t'], 'eval_seconds': ev['eval_seconds'],
  'benchmark': bm['benchmark'], 'peak_gpu_mem_GB': bm.get('peak_gpu_mem_reserved_GB'),
  'versions': {k: bm['env'].get(k) for k in ['torch','torchvision','transformers','opencv','python']},
}
summary['GO'] = bool(abs(r1 - 46.9) <= 1.0 and hours <= 6)
os.makedirs(f'{REPO}/experiments/results', exist_ok=True)
for path in [f'{DRIVE_ROOT}/go_no_go.json', f'{REPO}/experiments/results/go_no_go.json']:
    json.dump(summary, open(path, 'w'), indent=2, default=float)
print(json.dumps(summary, indent=2, default=float))
print('\nDECISION:', 'GO ✅' if summary['GO'] else 'NO-GO ⚠️  -> see docs/ROADMAP.md Fallbacks')


## After running
1. Download `experiments/results/go_no_go.json` (or copy from Drive `cv_hw2/go_no_go.json`) and commit it in VSCode (`experiments/results/go_no_go.json`).
2. Paste the printed summary into `docs/HANDOFF.md` (Status section) so Claude/Codex can plan the next step.
3. These numbers (GPU, memory, timings) feed the Proposal's *Dataset & compute* section.